In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data_utils

In [2]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
from helpers.openml_data_v2 import get_data, get_data1, our_new_list
from helpers.progress_bar import ProgressBar
from helpers.persistence import save_var, load_var

from copy import deepcopy

In [4]:
N_BOOTSTRAP = 30
PRE_EPOCHS = 1000
FINE_EPOCHS = 200
BATCH_SIZE = 128
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
UPDATE_INTERVAL = 50

In [5]:
class BasicDnnAe(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # encoder (f)
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_size),
            nn.ReLU(),
        )

    # pre training
    def forward(self, x):
        e = self.encoder(x)
        d = self.decoder(e)
        return d

In [6]:
class BasicDnn(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # encoder (f)
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.classification_head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            # nn.Dropout(0.20),
            nn.Linear(32, output_size),
            # # nn.Dropout(0.20),
        )

    # pre training
    def forward(self, x):
        e = self.encoder(x)
        c = self.classification_head(e)
        return c

In [7]:
def pretrain(model, 
               train_x, 
               val_x,
               pbar = False
              ):
    

    
    # transform them to tensors on GPU
    train_dataset = torch.from_numpy(train_x).float().to(DEVICE)
    
    valid_dataset = torch.from_numpy(val_x).float().to(DEVICE)
    
    train_loader = data_utils.DataLoader(dataset=train_dataset,
                                          batch_size = BATCH_SIZE,
                                          shuffle=True)
    valid_data_loader = data_utils.DataLoader(dataset=valid_dataset,
                                          batch_size = BATCH_SIZE,
                                          shuffle=True)
    
    # model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    train_lle = []
    valid_lle = []
    min_val_loss = np.inf
    best_state = model.state_dict()
    
    for epoch in range(PRE_EPOCHS):
        batch_losses = []
        batch_val_losses = []
        train_samples = 0
        valid_samples = 0
        
        model = model.train()
        for train in train_loader:
            optimizer.zero_grad()
            prediction = model(train)
            mse_loss = criterion(prediction, train)
            mse_loss.backward()
            optimizer.step()
            
            batch_losses.append(
                mse_loss.cpu().item() * train.shape[0])
            train_samples += train.shape[0]
            
            
        # valid every training epoch
        model = model.eval()
        for valid in valid_data_loader:
            with torch.no_grad():
                pred_valid = model(valid)
            valid_loss = criterion(pred_valid, valid).item()

            batch_val_losses.append(valid_loss*valid.shape[0])
            valid_samples += valid.shape[0]
            
            
        mean_loss = np.sum(batch_losses) / train_samples
        mean_val_loss = np.sum(batch_val_losses) / valid_samples
        
        # save the model with minimum loss
        if mean_val_loss < min_val_loss:
            min_val_loss = mean_val_loss
            best_epoch = epoch
            best_state = model.state_dict()
            
        
        pbar and epoch%UPDATE_INTERVAL==0 and pbar.set_description((
            f'epoch: {epoch}/{PRE_EPOCHS};'
            f'mean_loss: {mean_loss:.3f} ; '
            f'mean_val_loss: {mean_val_loss:.3f} (min: {min_val_loss:.3f}); '
        ))
        
        train_lle.append(mean_loss)
        valid_lle.append(mean_val_loss)
        
    
    model.load_state_dict(best_state)
    
    return model, (train_lle, valid_lle)
    

In [8]:
def finetune_and_test(model, 
               train_x, train_y,
               val_x, val_y, 
               test_x, test_y,
               pbar = False
              ):
    ''' train, val, test are expected to tuples of NumPy array'''
    
    # transform them to tensors on GPU
    train_dataset = data_utils.TensorDataset(
        torch.from_numpy(train_x).float().to(DEVICE),
        torch.from_numpy(train_y).long().to(DEVICE)
    )
    
    valid_dataset = data_utils.TensorDataset(
        torch.from_numpy(val_x).float().to(DEVICE),
        torch.from_numpy(val_y).long().to(DEVICE)
    )
    
    train_loader = data_utils.DataLoader(dataset=train_dataset,
                                          batch_size = BATCH_SIZE,
                                          shuffle=True)
    valid_data_loader = data_utils.DataLoader(dataset=valid_dataset,
                                          batch_size = BATCH_SIZE,
                                          shuffle=True)
    
    # model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    train_lle = []
    valid_lle = []
    min_val_loss = np.inf
    best_state = model.state_dict()
    
    for epoch in range(FINE_EPOCHS):
        batch_losses = []
        batch_val_losses = []
        train_samples = 0
        valid_samples = 0
        
        model = model.train()
        for train, target in train_loader:
            optimizer.zero_grad()
            prediction = model(train)
            classification_loss = criterion(prediction, target)
            classification_loss.backward()
            optimizer.step()
            
            batch_losses.append(
                classification_loss.cpu().item() * train.shape[0])
            train_samples += train.shape[0]
            
            
        # valid every training epoch
        model = model.eval()
        for batch_idx, (valid, target) in enumerate(valid_data_loader):
            with torch.no_grad():
                pred_valid = model(valid)
            valid_loss = criterion(pred_valid, target).item()

            batch_val_losses.append(valid_loss*valid.shape[0])
            valid_samples += valid.shape[0]
            
            
        mean_loss = np.sum(batch_losses) / train_samples
        mean_val_loss = np.sum(batch_val_losses) / valid_samples
        
        # save the model with minimum loss
        if mean_val_loss < min_val_loss:
            min_val_loss = mean_val_loss
            best_epoch = epoch
            best_state = model.state_dict()
            
        
        pbar and epoch%UPDATE_INTERVAL==0 and pbar.set_description((
            f'epoch: {epoch}/{FINE_EPOCHS};'
            f'mean_loss: {mean_loss:.3f} ; '
            f'mean_val_loss: {mean_val_loss:.3f} (min: {min_val_loss:.3f}); '
        ))
        
        train_lle.append(mean_loss)
        valid_lle.append(mean_val_loss)
        
    
    model.load_state_dict(best_state)
    
    
    test = torch.from_numpy(test_x).float().to(DEVICE)
    
    model = model.eval()
    with torch.no_grad():
        pred_test = model(test).argmax(1).cpu().detach().numpy()
    test_accuracy = f1_score(test_y, pred_test, average='weighted')
    
    return test_accuracy, (train_lle, valid_lle)
    

In [9]:
def save_curves(training_curves, finetuning_curves, filename='test-dnn-x.pdf'):
    with PdfPages(filename) as pdf:  
        # plot pretraining loss per fold
        for fold_idx, (train_loss, val_loss) in enumerate(training_curves):
            fig = plt.figure(figsize=(9, 6))
            x = np.arange(len(train_loss))
            plt.plot(x, train_loss, label='train')
            plt.plot(x, val_loss, label='val')
            plt.xlabel('epoch',fontsize=13)
            plt.ylabel('mean mse loss',fontsize=13)
            plt.legend(prop={'size': 13})
            plt.title(f'Pretraining fold={fold_idx};')
            pdf.savefig(fig)
            plt.close()
            
        # plot finetuning loss per fold
        for fold_idx, (train_loss, val_loss) in enumerate(finetuning_curves):
            fig = plt.figure(figsize=(9, 6))
            x = np.arange(len(train_loss))
            plt.plot(x, train_loss, label='train')
            plt.plot(x, val_loss, label='val')
            plt.xlabel('epoch',fontsize=13)
            plt.ylabel('mean cross-entropy loss',fontsize=13)
            plt.legend(prop={'size': 13})
            plt.title(f'Finetuning fold={fold_idx};')
            pdf.savefig(fig)
            plt.close()

In [10]:
# X_ohe, y = get_data(469)

# X_ohe
# train_x, test_x, train_y, test_y = train_test_split(X_ohe, y, 
#                                       stratify=y,
#                                       shuffle=True,
#                                       test_size=1/5,
#                                       random_state=0)

# train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y,
#                                                        stratify=train_y,
#                                                        test_size=1/8,
#                                                        random_state=42)

# scaler = StandardScaler().fit(train_x)
# train_x, valid_x, test_x = (scaler.transform(x) for x in [train_x, valid_x, test_x])


In [11]:
def simulate(dataset_id, pbar = False):
    
    X_ohe, y = get_data(dataset_id)
    input_size = X_ohe.shape[1]
    output_size = np.unique(y).shape[0]
    
    all_training_curves = []
    all_finetuning_curves = []
    all_test_scores = []
    pbar and pbar.add_prefix(f'starting')
    
    for fold_counter in range(N_BOOTSTRAP):
        pbar.clear_prefix()
        pbar and pbar.add_prefix(str(dataset_id))
        pbar and pbar.add_prefix(f'Bootstrap: {fold_counter+1}/{N_BOOTSTRAP}')
        
        train_x, test_x, train_y, test_y = train_test_split(X_ohe, y, 
                                              stratify=y,
                                              shuffle=True,
                                              test_size=1/5,
                                              random_state=fold_counter)

        train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y,
                                                               stratify=train_y,
                                                               test_size=1/8,
                                                               random_state=42)
        
        
        # standardize based on train features
        scaler = StandardScaler().fit(train_x)
        train_x, valid_x, test_x = (scaler.transform(x) for x in [train_x, valid_x, test_x])
    
        
        pbar.add_prefix('Pretraining')
        model = BasicDnnAe(input_size).to(DEVICE)
        pretrained_model, training_curves = pretrain(model, train_x, valid_x, pbar)
        
        pretrained_state = pretrained_model.encoder.state_dict()
        
        
        pbar.edit_last_prefix('Finetuning')
        model = BasicDnn(input_size, output_size).to(DEVICE)
        model.encoder.load_state_dict(deepcopy(pretrained_state))
        
        test_f1, finetuning_curves = finetune_and_test(model,
                                              train_x, train_y,
                                              valid_x, valid_y,
                                              test_x, test_y, pbar)
        
        all_test_scores.append(test_f1)
        all_training_curves.append(training_curves)
        all_finetuning_curves.append(finetuning_curves)
        
        
    save_curves(all_training_curves, all_finetuning_curves,  f'./generated/test-dnnAe-{dataset_id}.pdf')
    return all_test_scores

In [12]:
expt = 'test-dnnae'

save_path, export_path = f'./saved_vars/{expt}.pkl', f'./exports/{expt}.csv'

dataset_results = load_var(save_path) or {}
# dataset_results = {}

In [13]:
from time import time
start_time, end_time, time_taken = 0, 0, 0

In [14]:
names = [ 1063,  1510,  1464,   469,   458,  
         1494,  1068,  1049,    23, 1050, 
         40975, 40982,  1067,  1487,  1485,  
         4134, 40701,  1497, 1475,  4538]
# names = [1510]
names = our_new_list + [46]
pbar = ProgressBar(names)

for d in pbar:
    pbar.clear_prefix()
    
    if d in dataset_results.keys(): continue
    
    start_time = time()
    scores = simulate(d, pbar=pbar)
    end_time = time()
    print('time taken: ', end_time - start_time)
    
    dataset_results[d] = scores
    save_var(dataset_results, save_path)
    

46 | Bootstrap: 30/30 | Finetuning epoch: 150/200;mean_loss: 0.001 ; mean_val_loss: 0.267 (min: 0.130); : 

time taken:  4868.188687086105


In [15]:
(end_time - start_time) / 30 

162.2729562362035

In [16]:
import pandas as pd 

cols = [ 'dataset', 'corruption', 
        'fold', 'test_score']
rows = []

for k,v in dataset_results.items():
    for i,s in enumerate(v): rows.append([k, 'DNN-AE', i, s])
        
df = pd.DataFrame(rows, columns=cols)
df.to_csv(export_path)